# Code for doing the different clustering methods, as well as data preprocessing

In this file we run the different methods

### DCEC - Deep Convolutional Embedded Clustering
First load the data

In [ ]:
#  Loading the data with all 400 features (for 400x400 data) - alternative loading method

from scipy.io import loadmat
import torch
import numpy as np

filepath = "Input Data\OldAge_combined_run2_400_noZ_clean.mat"
fc_mat = loadmat(filepath)
fc_mat = fc_mat['run2_data']  # Extract the FC2 variable from the loaded .mat file

print(fc_mat.shape)  # Should print (400, 400, N_subjects) for FC2
 # prints (N_subjects, 400, 400) for collected matrices

# fc_mat_inv = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 400, 400)
# fc_mat_inv = fc_mat_inv[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 400, 400)

fc_mat_inv = fc_mat[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 400, 400)
print(fc_mat_inv.shape)  # Should print (N_subjects, 1, 400, 400)

X = torch.from_numpy(fc_mat_inv).float()  # Convert to PyTorch tensor
print("Data shape:", X.shape)  # Should print (N_subjects, 1, 400, 400)

# Expected shape is (Batch size, channels, height, width)

In [1]:
#  Loading the data with all 1000 features (for 1000x1000 data) - alternative loading method

from scipy.io import loadmat
import torch
import numpy as np

filepath = "My_FC_matrices\\FC_Y_run_1.mat"
fc_mat = loadmat(filepath)
print(fc_mat.keys())  # Print the keys to verify the structure of the loaded .mat file
fc_mat = fc_mat['run1_data']  # Extract the FC2 variable from the loaded .mat file

print(fc_mat.shape)  # Should print (1000, 1000, N_subjects) for FC2
 # prints (N_subjects, 1000, 1000) for collected matrices

# fc_mat_inv = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 1000, 1000)
# fc_mat_inv = fc_mat_inv[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 1000, 1000)

fc_mat_inv = fc_mat[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 1000, 1000)
print(fc_mat_inv.shape)  # Should print (N_subjects, 1, 1000, 1000)

X = torch.from_numpy(fc_mat_inv).float()  # Convert to PyTorch tensor
print("Data shape:", X.shape)  # Should print (N_subjects, 1, 1000, 1000)

# Expected shape is (Batch size, channels, height, width)

dict_keys(['__header__', '__version__', '__globals__', 'subject_names', 'run1_data', 'fc_matrices'])
(72, 1000, 1000)
(72, 1, 1000, 1000)
Data shape: torch.Size([72, 1, 1000, 1000])


Define the autoencoder specs for the Convolutional Autoencoder (CAE)

In [2]:
# Names
# 400x400
# name = "DCEC_400x400_72_O_subjects_run1"

# 1000x1000
name = "DCEC_1000x1000_72_O_subjects_run1"

In [5]:
# Config for 400x400 data
from Convolutional_AE import DCECConfig
cfg = DCECConfig(
    name=name,
    n_clusters=2,
    latent_dim=10,
    alpha=1.0,
    gamma=0.1,
    conv_layers_sizes=[1, 32, 64, 128, 256],
    epochs_pretrain=50,
    epochs_dcec=100,
    lr_pretrain=1e-3,
    lr_dcec=1e-3,
    update_interval=8, # Update target distribution every 4 batches, 2 batches in 72 subjects
    tol=4e-2, # 4e-2 = 0.04
    print_interval=10
)


Run the DCEC Model - single run

In [6]:
# Run the DCEC model once for the 200x200 data
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from torch.utils.data import TensorDataset, DataLoader

from data_loader import set_seed

# ----------------------------------------------------------
# DCEC - Deep Convolutional Embedded Clustering
# ----------------------------------------------------------
set_seed(42)

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=3, shuffle=False)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DCEC(cfg=cfg).to(device)

pretrain_cae(model, dataloader, device, epochs=cfg.epochs_pretrain, lr=cfg.lr_pretrain, print_interval=cfg.print_interval)
y_pred_initial = initialize_cluster_centers(model, dataloader, device)
print("Early cluster centers initialized.", y_pred_initial)


# Saving the model
torch.save(model.state_dict(), f"Models\\pre_train_model_{cfg.name}.pt") 


[Pretrain] Epoch 001/50 - Recon loss: 0.133266
[Pretrain] Epoch 005/50 - Recon loss: 0.059544
[Pretrain] Epoch 010/50 - Recon loss: 0.054870
[Pretrain] Epoch 015/50 - Recon loss: 0.049834
[Pretrain] Epoch 020/50 - Recon loss: 0.047736
[Pretrain] Epoch 025/50 - Recon loss: 0.049423
[Pretrain] Epoch 030/50 - Recon loss: 0.044245
[Pretrain] Epoch 035/50 - Recon loss: 0.051574
[Pretrain] Epoch 040/50 - Recon loss: 0.045124
[Pretrain] Epoch 045/50 - Recon loss: 0.042552
[Pretrain] Epoch 050/50 - Recon loss: 0.041284
Early cluster centers initialized. [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1
 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
# Run the DCEC model once for the 200x200 data
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from torch.utils.data import TensorDataset, DataLoader

from data_loader import set_seed

# ----------------------------------------------------------
# DCEC - Deep Convolutional Embedded Clustering
# ----------------------------------------------------------
set_seed(42)

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=3, shuffle=False)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DCEC(cfg=cfg).to(device)

pretrain_cae(model, dataloader, device, epochs=cfg.epochs_pretrain, lr=cfg.lr_pretrain, print_interval=cfg.print_interval)
y_pred_initial = initialize_cluster_centers(model, dataloader, device)
print("Early cluster centers initialized.", y_pred_initial)



train_dcec(
    model,
    dataloader,
    device,
    gamma=cfg.gamma,
    epochs=cfg.epochs_dcec,
    lr=cfg.lr_dcec,
    update_interval=cfg.update_interval,
    tol=cfg.tol,
    print_interval=cfg.print_interval
    )

q_final, labels = predict_soft_assignments(model, dataloader, device, save=True)
print("Predicted cluster labels:", labels)

# Saving the model
torch.save(model.state_dict(), f"Models\\model_{cfg.name}.pt") 


[Pretrain] Epoch 001/50 - Recon loss: 0.106030
[Pretrain] Epoch 005/50 - Recon loss: 0.060394
[Pretrain] Epoch 010/50 - Recon loss: 0.052425
[Pretrain] Epoch 015/50 - Recon loss: 0.048705


KeyboardInterrupt: 

In [4]:
# Run the DCEC model once for the 200x200 data
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from torch.utils.data import TensorDataset, DataLoader

from data_loader import set_seed

# ----------------------------------------------------------
# DCEC - Deep Convolutional Embedded Clustering
# ----------------------------------------------------------
set_seed(42)

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=3, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DCEC(cfg=cfg).to(device)
model.load_state_dict(torch.load(f"Models\\pre_train_model_{cfg.name}.pt"))  # Load the pre-trained model weights


train_dcec(
    model,
    dataloader,
    device,
    gamma=cfg.gamma,
    epochs=cfg.epochs_dcec,
    lr=cfg.lr_dcec,
    update_interval=cfg.update_interval,
    tol=cfg.tol,
    print_interval=cfg.print_interval
    )

q_final, labels = predict_soft_assignments(model, dataloader, device, save=True)
print("Predicted cluster labels:", labels)

# Saving the model
torch.save(model.state_dict(), f"Models\\model_{cfg.name}.pt") 


[DCEC] Epoch 001/100 - Total: 0.055104, Recon: 0.045159, KL: 0.009946
Stopping early: cluster assignments stabilized. Delta: 0.017157592 < Tol: 0.04
Stopped at epoch 3 and batch 16 after 64 updates.
Predicted cluster labels: [2 2 0 2 2 1 1 2 1 2 1 1 2 1 1 1 2 1 0 1 2 1 1 1 1 1 1 1 2 1 1 0 1 2 0 1 0
 2 1 2 1 2 2 1 2 1 1 2 1 2 2 2 1 2 0 1 1 2 0 2 1 1 1 1 1 2 1 1 1 1 1 1]


Evaluate the clustering from a single run

In [ ]:
# Evaluate clustering performance of a single run
from Evaluate_models import evaluate_single_clustering

triu_idx = np.triu_indices(200, k=1)
fc_mat_transp = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)

Features = np.array([
    fc_mat_transp[i][triu_idx] for i in range(fc_mat_transp.shape[0])
]) 
predictions = labels

z = model.z
print("z shape:", z.shape)
print("predictions shape:", predictions.shape)

evaluate_single_clustering(Features, predictions)
evaluate_single_clustering(z, predictions)


In [ ]:
# Plot the model from one run
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from Evaluate_models import plot_scores, Calculate_clustering_scores_from_folder
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=34, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg.n_clusters = 3  # Set the number of clusters to 3 for the latest model
model = DCEC(cfg=cfg).to(device)
model.load_state_dict(torch.load(f"Models/model_DCEC_200x200_clusters_3.pt"))  #


# Code for visualising the clustering results 
from Convolutional_AE import plot_training_history, plot_clustering, plot_reconstruction

plot_reconstruction(model, dataloader, device)
# plot_training_history(model)
plot_clustering(model, dataloader, device)

results = Calculate_clustering_scores_from_folder(models_dir="Clusters", model_prefix="DCEC_200x200_cluster", labels_tag="_labels_predicted_labels_", middle_layer_tag="_middle_layer_predicted_labels_")
plot_scores(results)




Run the DCEC model multiple times

In [8]:
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy
from data_loader import set_seed

# ----------------------------------------------------------
# DCEC - Deep Convolutional Embedded Clustering
# ----------------------------------------------------------

set_seed(42)  # Set a fixed random seed for reproducibility

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=34, shuffle=False) # Batch_size 34 gives 2 bacthes for 72 subjects. 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cluster_sizes =  list(range(2, 11)) # Example cluster sizes to evaluate

for n_clusters in cluster_sizes:
    run_cfg = deepcopy(cfg)  # Create a copy of the config for each run
    run_cfg.n_clusters = n_clusters
    
    model = DCEC(cfg=run_cfg).to(device)
    print(f"Running DCEC for cluster {n_clusters} ...")
    pretrain_cae(
        model,
        dataloader,
        device,
        epochs=run_cfg.epochs_pretrain,
        lr=run_cfg.lr_pretrain,
        print_interval=run_cfg.print_interval
    )

    # Saving the model
    torch.save(model.state_dict(), f"Models\\pre_trained_model_{cfg.name}_Cluster_{n_clusters}.pt") 


Running DCEC for cluster 2 ...
[Pretrain] Epoch 001/50 - Recon loss: 0.174140
[Pretrain] Epoch 005/50 - Recon loss: 0.085387
[Pretrain] Epoch 010/50 - Recon loss: 0.072242
[Pretrain] Epoch 015/50 - Recon loss: 0.065263
[Pretrain] Epoch 020/50 - Recon loss: 0.061342
[Pretrain] Epoch 025/50 - Recon loss: 0.057286
[Pretrain] Epoch 030/50 - Recon loss: 0.055640
[Pretrain] Epoch 035/50 - Recon loss: 0.054261
[Pretrain] Epoch 040/50 - Recon loss: 0.053476
[Pretrain] Epoch 045/50 - Recon loss: 0.052626
[Pretrain] Epoch 050/50 - Recon loss: 0.053223
Running DCEC for cluster 3 ...
[Pretrain] Epoch 001/50 - Recon loss: 0.092210
[Pretrain] Epoch 005/50 - Recon loss: 0.085165
[Pretrain] Epoch 010/50 - Recon loss: 0.072245
[Pretrain] Epoch 015/50 - Recon loss: 0.067891
[Pretrain] Epoch 020/50 - Recon loss: 0.063480
[Pretrain] Epoch 025/50 - Recon loss: 0.059585
[Pretrain] Epoch 030/50 - Recon loss: 0.056101
[Pretrain] Epoch 035/50 - Recon loss: 0.054289
[Pretrain] Epoch 040/50 - Recon loss: 0.05270

In [ ]:
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy
from data_loader import set_seed

# ----------------------------------------------------------
# DCEC - Deep Convolutional Embedded Clustering
# ----------------------------------------------------------

set_seed(42)  # Set a fixed random seed for reproducibility

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=34, shuffle=False) # Batch_size 34 gives 2 bacthes for 72 subjects. 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cluster_sizes =  list(range(2, 11)) # Example cluster sizes to evaluate

for n_clusters in cluster_sizes:
    run_cfg = deepcopy(cfg)  # Create a copy of the config for each run
    run_cfg.n_clusters = n_clusters
    
    model = DCEC(cfg=run_cfg).to(device)
    print(f"Running DCEC for cluster {n_clusters} ...")
    pretrain_cae(
        model,
        dataloader,
        device,
        epochs=run_cfg.epochs_pretrain,
        lr=run_cfg.lr_pretrain,
        print_interval=run_cfg.print_interval
    )
    y_pred_initial = initialize_cluster_centers(model, dataloader, device)
    print("Early cluster centers initialized.", y_pred_initial)

    train_dcec(
        model,
        dataloader,
        device,
        gamma=run_cfg.gamma,
        epochs=run_cfg.epochs_dcec,
        lr=run_cfg.lr_dcec,
        update_interval=run_cfg.update_interval,
        tol=run_cfg.tol,
        print_interval=run_cfg.print_interval,
        # y_pred_initial=y_pred_initial
    )

    q_final, labels = predict_soft_assignments(model, dataloader, device, save=True)
    print("Predicted cluster labels:", labels)

    # Saving the model
    torch.save(model.state_dict(), f"Models\\model_{cfg.name}_Cluster_{n_clusters}.pt") 


In [9]:
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy
from data_loader import set_seed

# ----------------------------------------------------------
# DCEC - Deep Convolutional Embedded Clustering
# ----------------------------------------------------------

set_seed(42)  # Set a fixed random seed for reproducibility

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=34, shuffle=False) # Batch_size 34 gives 2 bacthes for 72 subjects. 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cluster_sizes =  list(range(2, 11)) # Example cluster sizes to evaluate

for n_clusters in cluster_sizes:
    run_cfg = deepcopy(cfg)  # Create a copy of the config for each run
    run_cfg.n_clusters = n_clusters
    
    model = DCEC(cfg=run_cfg).to(device)
    print(f"Running DCEC for cluster {n_clusters} ...")
    
    # Load the pre-trained model weights
    model.load_state_dict(torch.load(f"Models\\pre_trained_model_{cfg.name}_Cluster_{n_clusters}.pt"))  # Load the pre-trained model weights
    y_pred_initial = initialize_cluster_centers(model, dataloader, device)
    print("Early cluster centers initialized.", y_pred_initial)

    train_dcec(
        model,
        dataloader,
        device,
        gamma=run_cfg.gamma,
        epochs=run_cfg.epochs_dcec,
        lr=run_cfg.lr_dcec,
        update_interval=run_cfg.update_interval,
        tol=run_cfg.tol,
        print_interval=run_cfg.print_interval,
        # y_pred_initial=y_pred_initial
    )

    q_final, labels = predict_soft_assignments(model, dataloader, device, save=True)
    print("Predicted cluster labels:", labels)

    # Saving the model
    torch.save(model.state_dict(), f"Models\\model_{cfg.name}_Cluster_{n_clusters}.pt") 


Running DCEC for cluster 2 ...
Early cluster centers initialized. [0 0 0 0 0 1 1 0 1 0 1 1 0 1 1 1 0 1 0 1 0 1 1 1 1 1 1 1 0 1 1 0 1 0 0 1 0
 0 1 0 1 0 0 1 0 1 1 0 1 0 0 0 1 0 0 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1]
[DCEC] Epoch 001/100 - Total: 0.104940, Recon: 0.088355, KL: 0.016584
[DCEC] Epoch 010/100 - Total: 0.058898, Recon: 0.055894, KL: 0.003004
[DCEC] Epoch 020/100 - Total: 0.055496, Recon: 0.052858, KL: 0.002638
[DCEC] Epoch 030/100 - Total: 0.058246, Recon: 0.055183, KL: 0.003063
Stopping early: cluster assignments stabilized. Delta: 0.007128458 < Tol: 0.04
Stopped at epoch 38 and batch 1 after 112 updates.
Predicted cluster labels: [1 1 0 1 1 1 1 1 1 0 1 1 0 1 1 1 0 1 0 1 0 1 1 1 1 1 1 1 0 1 1 0 1 1 0 1 0
 0 1 1 1 1 0 1 0 1 1 1 1 1 1 0 1 1 0 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1]
Running DCEC for cluster 3 ...
Early cluster centers initialized. [2 2 1 2 2 0 0 2 0 1 2 0 2 0 0 0 2 0 1 2 2 0 0 0 0 0 0 0 2 0 0 1 0 2 1 0 1
 1 0 2 0 2 2 0 1 0 0 2 0 2 2 2 0 2 1 0 0 2 1 2 0 0 0 0 2 2 0 0 

Evaluate the models from multiple cluster amounts

In [ ]:
# Print the clustering scores for all the runs 
from Evaluate_models import evaluate_single_clustering, Calculate_clustering_scores_from_folder
import os
import numpy as np

model_prefix = "DCEC_200x200_72_"
labels_tag = "_labels_predicted_labels_"
middle_layer_tag = "_middle_layer_predicted_labels_"

# Load the middle layer and predicted labels from the saved model
# Open the folder "Models"
Calculate_clustering_scores_from_folder(
    model_prefix = model_prefix,
    print_results = True
)

# Silhouett score: Higher is better
# Calinski-Harabasz score: Lower is better
# Davies-Bouldin score: Higher is better
        


In [ ]:
# PLot the results for all the runs
from Evaluate_models import plot_scores, Calculate_clustering_scores_from_folder
from Convolutional_AE import plot_clustering, DCEC
from torch.utils.data import TensorDataset, DataLoader
import os
import torch

results = Calculate_clustering_scores_from_folder(models_dir="Clusters", model_prefix="DCEC_400x400_72_O_subjects_run1_", labels_tag="_labels_predicted_labels_", middle_layer_tag="_middle_layer_predicted_labels_")
plot_scores(results, sort_scores=True, save_path="Figures\\DCEC_400x400_72_O_subjects_run1_clustering")

# Plot the clustering results for the models
dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=3, shuffle=False)
for model in os.listdir("Models"):
    if model.startswith("model_DCEC_400x400_72_O_subjects_run1_"): # Only plot the models with the specified prefix
        print(f"\n Evaluating model: {model}")
        cluster_number = model.split("Cluster_")[1].split(".pt")[0]  # Extract the cluster number from the model name
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        cfg.n_clusters = int(cluster_number)  # Set the number of clusters to the extracted number
        temp_model = DCEC(cfg=cfg).to(device)   
        temp_model.load_state_dict(torch.load(f"Models/{model}"))
        
        plot_clustering(temp_model, dataloader, device, save_path =f"Figures\\{model}.png")


### UMAP

In [ ]:
from UMAP import UMAP

### HDBSCAN
Perform HDBSCAN on the data

In [ ]:
from My_HDBSCAN import hdbscan_clustering

hdbscan_clustering(Functional_connectivity_matrix=Functional_connectivity_matrix, save_labels=True)

### Evaluate clustering against behavioural data

In [3]:
from data_loader import load_whole_behavioural_data, combine_cca_mlr_pipeline_outputs, convert_numpy
from Evaluate_clusterings_with_CCA import full_cca_selection_subject_pipeline
from Evaluate_clusterings_with_MLR import full_mlr_selection_subject_pipeline
import json

# Load data
csv_path = "C:\\Users\\oddafo\\Downloads\\Combined_Beh_Cognitive.csv"
used_ids_path = "Dummy Test Data\\Included_subjects_Young.csv"
cluster_labels_path = "Clusters\\DCEC_400x400_72_Y_subjects_run1_"

df, included_variables = load_whole_behavioural_data(csv_path=csv_path, included_ids_path=used_ids_path, cluster_labels_path=cluster_labels_path)

# Define parameters
cluster_col = ""
subject_id_col = "Subject"
Young_run1_output_dict = {}

for cluster in range(2, 10):
    cluster_col = f"Cluster_{cluster}"
    print(f"\n\n=== Running pipelines for {cluster_col} ===")
    # Run pipelines
    print("Running CCA pipeline...")
    pipeline_output_cca = full_cca_selection_subject_pipeline(
        data=df,
        cluster_col=cluster_col,
        candidate_variables=included_variables,
        cv_splits=5,
        random_state=42,
        max_variables=6,
        min_improvement=0.001,
        subject_id_col=subject_id_col,
        print_results=False
    )
    print("Running MLR pipeline...")
    pipeline_output_mlr = full_mlr_selection_subject_pipeline(
        data=df,
        cluster_col=cluster_col,
        candidate_variables=included_variables,
        cv_splits=5,
        random_state=42,
        max_variables=10,
        min_improvement=0.001,
        subject_id_col=subject_id_col,
        print_results=False
    )

    # Combine results
    print("Combining results")
    combined_results = combine_cca_mlr_pipeline_outputs(pipeline_output_cca, pipeline_output_mlr)
    print(f"MLR final results: {combined_results['mlr_removed_subjects_results']['mean_accuracy']:.4f}")

    Young_run1_output_dict[cluster_col+"_results"] = combined_results

# Save results to JSON
with open("Results\\Young_run1_cluster_behavioural_results.json", "w") as f:
    json.dump(Young_run1_output_dict, f, indent=4, default=convert_numpy)



=== Running pipelines for Cluster_2 ===
Running CCA pipeline...


c:\Users\oddafo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")


Running MLR pipeline...
Combining results
MLR final results: 0.8143


=== Running pipelines for Cluster_3 ===
Running CCA pipeline...


c:\Users\oddafo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")


Running MLR pipeline...
Combining results
MLR final results: 0.7857


=== Running pipelines for Cluster_4 ===
Running CCA pipeline...


...

c:\Users\oddafo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")


Running MLR pipeline...


c:\Users\oddafo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\oddafo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\oddafo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\oddafo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\oddafo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_s

Combining results
MLR final results: 0.9429


=== Running pipelines for Cluster_9 ===
Running CCA pipeline...


ValueError: A has a NaN entry